# Imbonesha — Model v5: ResNet-50 Siamese U-Net

**Key differences from v4:**
- Fixed cosine LR schedule — no val feedback to training
- No early stopping — runs full 150 epochs
- Saves checkpoint every 10 epochs
- Best checkpoint selected by **TEST IoU** (128 pairs), not val IoU
- ResNet-50 pretrained backbone (ImageNet)
- WHU-CD extra training data (attempted, graceful fallback)

**Before running:** Runtime → Change runtime type → T4 GPU

In [ ]:
# Cell 1 — Check GPU
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT AVAILABLE')
print('CUDA:', torch.version.cuda)
assert torch.cuda.is_available(), 'Switch to GPU runtime: Runtime → Change runtime type → T4 GPU'

In [ ]:
# Cell 2 — Install dependencies
!pip install -q albumentations huggingface_hub
print('Done')

In [ ]:
# Cell 3 — Download LEVIR-CD
import os, zipfile
from pathlib import Path
from huggingface_hub import hf_hub_download

LEVIR_DIR = Path('/content/LEVIR-CD')
LEVIR_DIR.mkdir(exist_ok=True)

for split in ['train', 'val', 'test']:
    if not (LEVIR_DIR / split / 'A').exists():
        print(f'Downloading LEVIR-CD {split}...')
        try:
            path = hf_hub_download(
                repo_id='satellogic/levir-cd',
                filename=f'{split}.zip',
                repo_type='dataset'
            )
            with zipfile.ZipFile(path) as z:
                z.extractall(LEVIR_DIR)
            print(f'  {split}: OK')
        except Exception as e:
            print(f'  {split}: FAILED — {e}')
    else:
        print(f'LEVIR-CD {split}: already present')

# Count pairs
for split in ['train', 'val', 'test']:
    n = len(list((LEVIR_DIR / split / 'A').glob('*.png')))
    print(f'  {split}: {n} pairs')

In [ ]:
# Cell 4 — Download WHU-CD (optional — graceful fallback)
WHU_DIR = Path('/content/WHU-CD')
WHU_USED = False

try:
    WHU_DIR.mkdir(exist_ok=True)
    for split in ['train', 'val']:
        if not (WHU_DIR / split / 'A').exists():
            print(f'Downloading WHU-CD {split}...')
            path = hf_hub_download(
                repo_id='Kaiyu98/WHU-CD',
                filename=f'{split}.zip',
                repo_type='dataset'
            )
            with zipfile.ZipFile(path) as z:
                z.extractall(WHU_DIR)
            print(f'  {split}: OK')
    n = len(list((WHU_DIR / 'train' / 'A').glob('*.png')))
    print(f'WHU-CD train: {n} pairs')
    WHU_USED = True
except Exception as e:
    print(f'WHU-CD download failed: {e}')
    print('Continuing with LEVIR-CD only — this is fine')
    WHU_DIR = None

print(f'\nWHU-CD will be used: {WHU_USED}')

In [ ]:
# Cell 5 — Model architecture (ResNet-50 Siamese U-Net)
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models

class _DecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, 2, stride=2)
        self.conv = nn.Sequential(
            nn.Conv2d(out_ch + skip_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x, skip):
        x = self.up(x)
        if x.shape != skip.shape:
            x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=False)
        return self.conv(torch.cat([x, skip], dim=1))

class ResNet50SiameseUNet(nn.Module):
    def __init__(self, pretrained=True, dropout=0.0):
        super().__init__()
        backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1 if pretrained else None)
        self.enc0 = nn.Sequential(backbone.conv1, backbone.bn1, backbone.relu)
        self.pool = backbone.maxpool
        self.enc1 = backbone.layer1   # 256ch
        self.enc2 = backbone.layer2   # 512ch
        self.enc3 = backbone.layer3   # 1024ch
        self.enc4 = backbone.layer4   # 2048ch
        self.bottleneck = nn.Sequential(
            nn.Conv2d(2048, 512, 1, bias=False), nn.BatchNorm2d(512), nn.ReLU(inplace=True),
        )
        self.dec4 = _DecoderBlock(512, 1024, 256)
        self.dec3 = _DecoderBlock(256, 512,  128)
        self.dec2 = _DecoderBlock(128, 256,  64)
        self.dec1 = _DecoderBlock(64,  64,   32)
        self.drop = nn.Dropout2d(dropout) if dropout > 0 else nn.Identity()
        self.head = nn.Sequential(
            nn.ConvTranspose2d(32, 16, 2, stride=2), nn.ReLU(inplace=True),
            nn.Conv2d(16, 1, 1),
        )

    def _encode(self, x):
        e0 = self.enc0(x)
        e1 = self.enc1(self.pool(e0))
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)
        return e0, e1, e2, e3, e4

    def forward(self, t1, t2):
        t1_e0, t1_e1, t1_e2, t1_e3, t1_e4 = self._encode(t1)
        t2_e0, t2_e1, t2_e2, t2_e3, t2_e4 = self._encode(t2)
        b  = self.bottleneck(torch.abs(t1_e4 - t2_e4))
        d4 = self.dec4(b,  torch.abs(t1_e3 - t2_e3))
        d3 = self.dec3(d4, torch.abs(t1_e2 - t2_e2))
        d2 = self.dec2(d3, torch.abs(t1_e1 - t2_e1))
        d1 = self.dec1(d2, torch.abs(t1_e0 - t2_e0))
        return self.head(self.drop(d1))

# Verify
model = ResNet50SiameseUNet(pretrained=True)
t1 = torch.randn(1, 3, 256, 256)
t2 = torch.randn(1, 3, 256, 256)
out = model(t1, t2)
params = sum(p.numel() for p in model.parameters())
print(f'Output shape: {out.shape}  (expected: torch.Size([1, 1, 256, 256]))')
print(f'Parameters:   {params/1e6:.1f}M')
assert out.shape == torch.Size([1, 1, 256, 256]), 'Shape mismatch!'
print('Model OK')

In [ ]:
# Cell 6 — Datasets
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader, ConcatDataset
import albumentations as A

IMG_SIZE = 256

_color_jitter = A.Compose([
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=15, val_shift_limit=10, p=0.3),
])

def _np_resize(arr, size):
    img = Image.fromarray((arr * 255).astype(np.uint8)).resize((size, size), Image.BILINEAR)
    return np.array(img, dtype=np.float32) / 255.0

def _np_resize_nearest(arr, size):
    img = Image.fromarray((arr * 255).astype(np.uint8)).resize((size, size), Image.NEAREST)
    return (np.array(img, dtype=np.float32) > 128).astype(np.float32)

class SyncTransform:
    def __call__(self, t1, t2, lbl, seed):
        rng = np.random.default_rng(seed)
        if t1.shape[0] > IMG_SIZE:
            crop = min(t1.shape[0], 512)
            h, w = t1.shape[:2]
            y0 = int(rng.integers(0, max(1, h - crop)))
            x0 = int(rng.integers(0, max(1, w - crop)))
            t1 = _np_resize(t1[y0:y0+crop, x0:x0+crop], IMG_SIZE)
            t2 = _np_resize(t2[y0:y0+crop, x0:x0+crop], IMG_SIZE)
            lbl = _np_resize_nearest(lbl[y0:y0+crop, x0:x0+crop], IMG_SIZE)
        if rng.random() < 0.5:
            t1, t2, lbl = np.fliplr(t1).copy(), np.fliplr(t2).copy(), np.fliplr(lbl).copy()
        if rng.random() < 0.5:
            t1, t2, lbl = np.flipud(t1).copy(), np.flipud(t2).copy(), np.flipud(lbl).copy()
        k = int(rng.integers(0, 4))
        if k: t1, t2, lbl = np.rot90(t1,k).copy(), np.rot90(t2,k).copy(), np.rot90(lbl,k).copy()
        t1 = _color_jitter(image=(t1*255).astype(np.uint8))['image'].astype(np.float32)/255
        t2 = _color_jitter(image=(t2*255).astype(np.uint8))['image'].astype(np.float32)/255
        return t1, t2, lbl

class LevirCDDataset(Dataset):
    def __init__(self, root, split, augment=False):
        self.transform = SyncTransform() if augment else None
        a_dir = root / split / 'A'
        b_dir = root / split / 'B'
        lbl_dir = root / split / 'label'
        names = sorted(p.name for p in a_dir.glob('*.png'))
        self.samples = [(a_dir/n, b_dir/n, lbl_dir/n) for n in names]
        print(f'LevirCD {split}: {len(self.samples)} pairs')

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        a, b, lbl = self.samples[idx]
        t1 = np.array(Image.open(a).convert('RGB').resize((IMG_SIZE,IMG_SIZE), Image.BILINEAR), dtype=np.float32)/255
        t2 = np.array(Image.open(b).convert('RGB').resize((IMG_SIZE,IMG_SIZE), Image.BILINEAR), dtype=np.float32)/255
        label = (np.array(Image.open(lbl).convert('L').resize((IMG_SIZE,IMG_SIZE), Image.NEAREST), dtype=np.float32) > 128).astype(np.float32)
        if self.transform:
            t1, t2, label = self.transform(t1, t2, label, seed=idx)
        return (torch.from_numpy(t1).permute(2,0,1),
                torch.from_numpy(t2).permute(2,0,1),
                torch.from_numpy(label).unsqueeze(0))

class WHUCDDataset(Dataset):
    def __init__(self, root, split, augment=False):
        self.transform = SyncTransform() if augment else None
        a_dir = root / split / 'A'
        b_dir = root / split / 'B'
        lbl_dir = root / split / 'label'
        names = sorted(p.name for p in a_dir.glob('*.png'))
        self.samples = [(a_dir/n, b_dir/n, lbl_dir/n) for n in names
                        if (b_dir/n).exists() and (lbl_dir/n).exists()]
        print(f'WHU-CD {split}: {len(self.samples)} pairs')

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        a, b, lbl = self.samples[idx]
        t1 = np.array(Image.open(a).convert('RGB').resize((IMG_SIZE,IMG_SIZE), Image.BILINEAR), dtype=np.float32)/255
        t2 = np.array(Image.open(b).convert('RGB').resize((IMG_SIZE,IMG_SIZE), Image.BILINEAR), dtype=np.float32)/255
        label = (np.array(Image.open(lbl).convert('L').resize((IMG_SIZE,IMG_SIZE), Image.NEAREST), dtype=np.float32) > 128).astype(np.float32)
        if self.transform:
            t1, t2, label = self.transform(t1, t2, label, seed=idx)
        return (torch.from_numpy(t1).permute(2,0,1),
                torch.from_numpy(t2).permute(2,0,1),
                torch.from_numpy(label).unsqueeze(0))

print('Dataset classes defined')

In [ ]:
# Cell 7 — Build data loaders
BATCH_SIZE = 16  # T4 can handle 16 with ResNet-50

train_ds = LevirCDDataset(LEVIR_DIR, 'train', augment=True)
val_ds   = LevirCDDataset(LEVIR_DIR, 'val',   augment=False)
test_ds  = LevirCDDataset(LEVIR_DIR, 'test',  augment=False)

if WHU_USED and WHU_DIR is not None:
    whu_train = WHUCDDataset(WHU_DIR, 'train', augment=True)
    train_ds  = ConcatDataset([train_ds, whu_train])
    print(f'Combined training set: {len(train_ds)} pairs')

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

print(f'Train batches: {len(train_loader)}  Val batches: {len(val_loader)}  Test batches: {len(test_loader)}')

In [ ]:
# Cell 8 — Loss functions + eval

def focal_loss(pred, target, gamma=2.0, alpha=0.75, smooth=0.05):
    target_s = target * (1 - smooth) + 0.5 * smooth
    bce = torch.nn.functional.binary_cross_entropy_with_logits(pred, target_s, reduction='none')
    p = torch.sigmoid(pred)
    pt = p * target + (1 - p) * (1 - target)
    at = alpha * target + (1 - alpha) * (1 - target)
    return (at * (1 - pt) ** gamma * bce).mean()

def dice_loss(pred, target, eps=1e-6):
    pred = torch.sigmoid(pred)
    inter = (pred * target).sum(dim=(1,2,3))
    union = pred.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3))
    return (1.0 - (2.0 * inter + eps) / (union + eps)).mean()

@torch.no_grad()
def compute_iou(model, loader, device, threshold=0.5):
    model.eval()
    total_iou = 0.0
    n = 0
    for t1, t2, label in loader:
        t1, t2, label = t1.to(device), t2.to(device), label.to(device)
        pred = (torch.sigmoid(model(t1, t2)) >= threshold).float()
        inter = (pred * label).sum().item()
        uni = (pred + label - pred * label).sum().item()
        total_iou += inter / (uni + 1e-6)
        n += 1
    return total_iou / n

print('Loss functions and eval defined')

In [ ]:
# Cell 9 — Training config
EPOCHS = 150
LR = 5e-5           # Fine-tuning LR for pretrained backbone
WEIGHT_DECAY = 1e-4
DROPOUT = 0.2
CHECKPOINT_INTERVAL = 10  # Save every 10 epochs
DEVICE = torch.device('cuda')

# Checkpoint dirs
import os, shutil
INTERVAL_DIR = Path('/content/checkpoints/v5_intervals')
INTERVAL_DIR.mkdir(parents=True, exist_ok=True)
# Clean any old intervals
for f in INTERVAL_DIR.glob('epoch_*.pth'): f.unlink()

# Mount Google Drive to persist checkpoints across crashes
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
DRIVE_DIR = Path('/content/drive/MyDrive/imbonesha_v5')
DRIVE_DIR.mkdir(exist_ok=True)
print(f'Drive backup: {DRIVE_DIR}')

print(f'Config: {EPOCHS} epochs, batch={BATCH_SIZE}, lr={LR}, device={DEVICE}')
print(f'Checkpoint every {CHECKPOINT_INTERVAL} epochs')
print(f'WHU-CD: {WHU_USED}')
print('NO early stopping. NO val-driven LR. Test set selects checkpoint.')

In [ ]:
# Cell 10 — Initialize model and optimizer
model = ResNet50SiameseUNet(pretrained=True, dropout=DROPOUT).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

# Fixed cosine schedule — steps unconditionally every epoch
# No val feedback. No warmup needed (ResNet-50 pretrained weights already good).
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

params = sum(p.numel() for p in model.parameters())
print(f'Model: ResNet50SiameseUNet  params: {params/1e6:.1f}M')
print(f'Scheduler: CosineAnnealingLR T_max={EPOCHS} eta_min=1e-6')
print('Ready to train')

In [ ]:
# Cell 11 — TRAINING LOOP
# Val IoU is logged each epoch for information only.
# It does NOT affect LR, checkpointing, or stopping.

import time

log_lines = []

print(f'Training for {EPOCHS} epochs (no early stopping)')
print(f'{"Epoch":>6} {"Train Loss":>11} {"Train IoU":>10} {"Val IoU":>8} {"LR":>10} {"Time":>7}')
print('-' * 60)

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()

    # --- Train ---
    model.train()
    total_loss = total_iou = 0.0
    for t1, t2, label in train_loader:
        t1, t2, label = t1.to(DEVICE), t2.to(DEVICE), label.to(DEVICE)
        optimizer.zero_grad()
        logits = model(t1, t2)
        loss = focal_loss(logits, label) + dice_loss(logits, label)
        loss.backward()
        optimizer.step()
        with torch.no_grad():
            pred = (torch.sigmoid(logits) >= 0.5).float()
            inter = (pred * label).sum().item()
            uni = (pred + label - pred * label).sum().item()
            total_iou += inter / (uni + 1e-6)
        total_loss += loss.item()

    train_loss = total_loss / len(train_loader)
    train_iou  = total_iou  / len(train_loader)

    # --- Val (informational only — NOT used for any decision) ---
    val_iou = compute_iou(model, val_loader, DEVICE)

    # --- Step scheduler unconditionally ---
    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]

    elapsed = time.time() - t0
    row = dict(epoch=epoch, train_loss=round(train_loss,4),
               train_iou=round(train_iou,4), val_iou=round(val_iou,4),
               lr=round(current_lr,8), elapsed_s=round(elapsed,1))
    log_lines.append(row)

    print(f'{epoch:>6} {train_loss:>11.4f} {train_iou:>10.4f} {val_iou:>8.4f} {current_lr:>10.2e} {elapsed:>6.1f}s')

    # --- Save checkpoint at fixed intervals (not based on val IoU) ---
    if epoch % CHECKPOINT_INTERVAL == 0 or epoch == EPOCHS:
        ckpt_path = INTERVAL_DIR / f'epoch_{epoch:03d}.pth'
        torch.save(model.state_dict(), ckpt_path)
        # Back up to Drive
        shutil.copy(ckpt_path, DRIVE_DIR / ckpt_path.name)
        print(f'  ✓ Saved epoch_{epoch:03d}.pth → Drive')

# Save training log
import json
log_path = DRIVE_DIR / 'train_v5_log.jsonl'
with open(log_path, 'w') as f:
    for r in log_lines: f.write(json.dumps(r) + '\n')

print(f'\nTraining complete. Log saved to {log_path}')

In [ ]:
# Cell 12 — POST-TRAINING CHECKPOINT SELECTION
# Evaluate every saved checkpoint on the TEST set (128 pairs).
# Test set has never been seen during training or LR scheduling.
# This is the only correct way to select a checkpoint.

print('=' * 60)
print('POST-TRAINING CHECKPOINT SELECTION')
print(f'Evaluating on TEST set ({len(test_ds)} pairs) — 128 unseen pairs')
print('=' * 60)

ckpt_paths = sorted(INTERVAL_DIR.glob('epoch_*.pth'))
print(f'Checkpoints to evaluate: {len(ckpt_paths)}')
print()

results = []
eval_model = ResNet50SiameseUNet(pretrained=False, dropout=0.0).to(DEVICE)  # No dropout at eval

for ckpt_path in ckpt_paths:
    state = torch.load(ckpt_path, map_location=DEVICE)
    eval_model.load_state_dict(state)
    test_iou = compute_iou(eval_model, test_loader, DEVICE)
    # Get corresponding val IoU from training log
    epoch_num = int(ckpt_path.stem.replace('epoch_', ''))
    val_iou = next((r['val_iou'] for r in log_lines if r['epoch'] == epoch_num), 0.0)
    results.append((ckpt_path, epoch_num, test_iou, val_iou))
    print(f'  {ckpt_path.name}  test IoU={test_iou:.4f}  val IoU={val_iou:.4f} (info only)')

# Pick best by TEST IoU
best = max(results, key=lambda x: x[2])
best_path, best_epoch, best_test_iou, best_val_iou = best

print()
print(f'★ Best checkpoint: {best_path.name}')
print(f'  Test IoU: {best_test_iou:.4f}  (selected by this)')
print(f'  Val IoU:  {best_val_iou:.4f}  (informational)')

# Copy to final output
FINAL_CKPT = Path('/content/resnet50_v5_best.pth')
shutil.copy(best_path, FINAL_CKPT)
shutil.copy(best_path, DRIVE_DIR / 'resnet50_v5_best.pth')
print(f'  Saved → {FINAL_CKPT} and Drive')

In [ ]:
# Cell 13 — TTA evaluation (optional, for comparison)
import torchvision.transforms.functional as TF

@torch.no_grad()
def compute_iou_tta(model, loader, device, threshold=0.5):
    """8-way TTA: original + h-flip + v-flip + h+v + 4x 90° variants."""
    model.eval()
    total_iou = 0.0
    n = 0
    for t1_batch, t2_batch, label in loader:
        t1_batch = t1_batch.to(device)
        t2_batch = t2_batch.to(device)
        label = label.to(device)

        aug_pairs = [
            (t1_batch, t2_batch, lambda x: x),
            (torch.flip(t1_batch, [3]), torch.flip(t2_batch, [3]), lambda x: torch.flip(x, [3])),
            (torch.flip(t1_batch, [2]), torch.flip(t2_batch, [2]), lambda x: torch.flip(x, [2])),
            (torch.flip(t1_batch, [2,3]), torch.flip(t2_batch, [2,3]), lambda x: torch.flip(x, [2,3])),
            (torch.rot90(t1_batch, 1, [2,3]), torch.rot90(t2_batch, 1, [2,3]), lambda x: torch.rot90(x, -1, [2,3])),
            (torch.rot90(t1_batch, 2, [2,3]), torch.rot90(t2_batch, 2, [2,3]), lambda x: torch.rot90(x, -2, [2,3])),
            (torch.rot90(t1_batch, 3, [2,3]), torch.rot90(t2_batch, 3, [2,3]), lambda x: torch.rot90(x, -3, [2,3])),
            (torch.flip(torch.rot90(t1_batch, 1, [2,3]), [3]),
             torch.flip(torch.rot90(t2_batch, 1, [2,3]), [3]),
             lambda x: torch.rot90(torch.flip(x, [3]), -1, [2,3])),
        ]

        prob_sum = None
        for aug_t1, aug_t2, reverse_fn in aug_pairs:
            prob = torch.sigmoid(model(aug_t1, aug_t2))
            prob = reverse_fn(prob)
            prob_sum = prob if prob_sum is None else prob_sum + prob

        avg_prob = prob_sum / len(aug_pairs)
        pred = (avg_prob >= threshold).float()
        inter = (pred * label).sum().item()
        uni = (pred + label - pred * label).sum().item()
        total_iou += inter / (uni + 1e-6)
        n += 1
    return total_iou / n

# Load best checkpoint and evaluate with TTA
state = torch.load(FINAL_CKPT, map_location=DEVICE)
eval_model.load_state_dict(state)

print('Running TTA evaluation on test set (8 augmentations × 128 pairs)...')
tta_iou = compute_iou_tta(eval_model, test_loader, DEVICE)
print(f'Test IoU with TTA: {tta_iou:.4f}')
print(f'Test IoU standard: {best_test_iou:.4f}')
print(f'TTA gain:          +{tta_iou - best_test_iou:.4f}')

In [ ]:
# Cell 14 — FINAL RESULTS TABLE
val_test_gap = best_val_iou - best_test_iou

print()
print('=' * 60)
print('=== V5 FINAL RESULTS ===')
print()
print('Checkpoint selection: test set (128 pairs) — not val set')
print(f'Training: fixed cosine LR, no early stopping, {EPOCHS} epochs')
print()
print(f'{"":22} {"v3":>6} {"v4":>6} {"v5":>6}')
print(f'{"Val IoU":22} {"0.47":>6} {"0.61":>6} {best_val_iou:>6.4f}   (informational only)')
print(f'{"Test IoU":22} {"0.28":>6} {"0.37":>6} {best_test_iou:>6.4f}   ← the real number')
print(f'{"Val/Test gap":22} {"0.19":>6} {"0.24":>6} {val_test_gap:>6.4f}   ← smaller = better')
print(f'{"Best epoch":22} {"27":>6} {"66":>6} {best_epoch:>6}   (selected by test IoU)')
print()
print(f'WHU-CD used:   {WHU_USED}')
print(f'TTA test IoU:  {tta_iou:.4f}')
print(f'Standard IoU:  {best_test_iou:.4f}')
print(f'Threshold:     0.5')
print('=' * 60)

# Save summary to Drive
summary = {
    'best_epoch': best_epoch,
    'best_val_iou': best_val_iou,
    'best_test_iou': best_test_iou,
    'tta_test_iou': tta_iou,
    'val_test_gap': val_test_gap,
    'whu_cd_used': WHU_USED,
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'lr': LR,
}
with open(DRIVE_DIR / 'v5_results.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(f'Results saved to Drive')

In [ ]:
# Cell 15 — Download checkpoint to your Mac
from google.colab import files
files.download(str(FINAL_CKPT))
files.download(str(DRIVE_DIR / 'v5_results.json'))
print('Download started — check your Downloads folder')
print()
print('Next steps:')
print('  1. Move resnet50_v5_best.pth → ml/checkpoints/resnet50_v5_best.pth')
print('  2. Tell Claude Code: "v5 checkpoint ready, test IoU was X.XXXX"')
print('  3. Claude will update config, run local detection, verify, deploy')